# 🎓 Buổi 9/9 — Quiz Tốt Nghiệp: Dự Đoán Giá Nhà

### Ôn lại toàn bộ kiến thức 9 buổi — 20 câu trắc nghiệm

---

> **📍 Lộ trình đã hoàn thành:**
> ```
> Buổi 1 ✅  Giới thiệu dự án & lộ trình
> Buổi 2 ✅  Setup môi trường & khám phá SalePrice
> Buổi 3 ✅  EDA — Phân tích missing, tương quan, outliers
> Buổi 4 ✅  Data Cleaning & Preprocessing
> Buổi 5 ✅  Feature Engineering
> Buổi 6 ✅  Model Training (Linear + Tree + Stacking)
> Buổi 7 ✅  Model Evaluation (Đánh giá toàn diện)
> Buổi 8 ✅  Submit Kaggle
> Buổi 9 🎓  Quiz Tốt Nghiệp  ← BẠN ĐANG Ở ĐÂY
> ```

---

| Phần | Chủ đề | Số câu |
|------|--------|--------|
| **A** | EDA & Data Cleaning | Q1 – Q5 |
| **B** | Feature Engineering & Preprocessing | Q6 – Q10 |
| **C** | Model Training & Linear Models | Q11 – Q15 |
| **D** | Ensemble & Evaluation | Q16 – Q20 |

> 💡 **Cách làm:** Đọc câu hỏi → tự chọn đáp án → kéo xuống xem **✅ Đáp án & Giải thích** bên dưới mỗi câu.


---

## 📊 Phần A — EDA & Data Cleaning (Q1–Q5)

---

### ❓ Câu 1: Tại sao ta dùng `log1p(SalePrice)` thay vì SalePrice gốc khi train model?

- A) Vì phân phối SalePrice gốc bị lệch phải (right-skewed) — log giúp nó gần phân phối chuẩn hơn, cải thiện chất lượng model
- B) Vì log làm cho dữ liệu nhỏ hơn, tiết kiệm bộ nhớ RAM
- C) Vì các model như Ridge và Lasso chỉ hoạt động với giá trị dương nhỏ hơn 1,000
- D) Vì Kaggle yêu cầu nộp submission ở dạng log(SalePrice)

---

> ✅ **Đáp án: A**
>
> SalePrice gốc có phân phối lệch phải mạnh — phần lớn nhà giá thấp, một số ít nhà rất đắt kéo đuôi phải dài. Các model tuyến tính (Ridge, Lasso) giả định sai số có phân phối chuẩn → log1p giúp thỏa mãn điều này.
>
> `log1p(x) = log(1 + x)` nén phân phối lại:
> - Phân phối gần chuẩn hơn → model hoạt động tốt hơn
> - RMSE trên log ≈ sai số phần trăm trên giá gốc (đánh giá công bằng hơn)
>
> Sau khi dự đoán, dùng `np.expm1()` để chuyển ngược lại SalePrice gốc.

---

### ❓ Câu 2: Với cột `LotFrontage` (diện tích mặt tiền), cách fill missing nào phù hợp nhất?

- A) Fill bằng 0 — vì nhà không có mặt tiền thì diện tích = 0
- B) Xóa toàn bộ các dòng có `LotFrontage` bị missing
- C) Fill bằng median theo từng **Neighborhood** — nhà cùng khu vực có mặt tiền tương đồng
- D) Fill bằng median toàn bộ dataset

---

> ✅ **Đáp án: C**
>
> `LotFrontage` bị missing không có nghĩa là nhà không có mặt tiền (≠ 0). Mặt tiền phụ thuộc rất nhiều vào **vị trí địa lý** — nhà cùng khu phố (`Neighborhood`) thường có quy hoạch tương đồng.
>
> `groupby('Neighborhood').transform(lambda x: x.fillna(x.median()))` là cách thông minh nhất:
> - Chính xác hơn median toàn dataset (đáp án D)
> - Không mất dữ liệu như xóa dòng (đáp án B)
> - Có cơ sở domain knowledge thực tế

---

### ❓ Câu 3: Outlier nào sau đây CẦN xóa khỏi training data trước khi train model?

- A) Những ngôi nhà có `GrLivArea > 4,000 sqft` nhưng `SalePrice < $300,000`
- B) Những ngôi nhà có `OverallQual = 10` (chất lượng hoàn hảo, cao nhất)
- C) Những ngôi nhà trong khu `Neighborhood = NridgHt` (khu cao cấp nhất)
- D) Những ngôi nhà có `YearBuilt` trước năm 1900

---

> ✅ **Đáp án: A**
>
> Nhà có diện tích sống rất lớn (>4,000 sqft) nhưng giá rất thấp (<$300K) là **outlier bất thường** — chúng phá vỡ hoàn toàn mối quan hệ giữa diện tích và giá mà model cần học.
>
> Các trường hợp B, C, D đều là dữ liệu **hợp lệ và có lý** — xóa chúng sẽ làm mất thông tin quan trọng.
>
> **Nguyên tắc:** Xóa outlier phải có lý do rõ ràng từ **domain knowledge**, không phải vì giá trị "trông lạ". Ví dụ outlier A: rất có thể là giao dịch phi thương mại (bán nội bộ gia đình, tịch biên...) không phản ánh thị trường thực.

---

### ❓ Câu 4: Hệ số tương quan (correlation) giữa `OverallQual` và `SalePrice` là 0.79. Điều này có nghĩa là gì?

- A) Chất lượng nhà là **nguyên nhân** làm giá nhà tăng (quan hệ nhân quả trực tiếp)
- B) Có mối quan hệ tuyến tính mạnh cùng chiều — chất lượng càng cao, giá thường càng cao
- C) 79% giá nhà được giải thích hoàn toàn bởi chất lượng ngôi nhà
- D) Nếu `OverallQual` tăng 1 điểm thì `SalePrice` tăng đúng 79%

---

> ✅ **Đáp án: B**
>
> Correlation = 0.79 đo **mức độ liên hệ tuyến tính**, không phải nhân quả.
>
> **Các sai lầm phổ biến:**
> - A sai: Correlation ≠ Causation. Ngôi nhà chất lượng cao và giá cao có thể cùng bị ảnh hưởng bởi yếu tố thứ 3 (vị trí, diện tích đất...).
> - C sai: R² = 0.79² ≈ 0.62, tức chỉ ~62% variance được giải thích — không phải 79%.
> - D sai: Đó là ý nghĩa của **hệ số hồi quy β**, không phải correlation.

---

### ❓ Câu 5: Khi vẽ heatmap correlation, tại sao ta dùng `abs(correlation)` để sắp xếp features?

- A) Vì pandas không hỗ trợ sort giá trị âm trong DataFrame index
- B) Vì correlation âm thể hiện quan hệ không hợp lệ, cần loại bỏ trước khi vẽ
- C) Vì tương quan âm với SalePrice nghĩa là feature đó hoàn toàn vô ích
- D) Vì cả tương quan dương (+) lẫn âm (−) đều mang thông tin — giá trị tuyệt đối đo "sức mạnh" bất kể chiều

---

> ✅ **Đáp án: D**
>
> Tương quan âm với SalePrice cũng rất có giá trị. Ví dụ: `HouseAge` (nhà càng cũ giá càng thấp) có correlation ≈ −0.55 — mang thông tin quan trọng không kém tương quan dương.
>
> Dùng `abs()` để **đo "sức mạnh" quan hệ** bất kể chiều hướng, giúp xác định đúng features quan trọng nhất đưa vào model.
>
> Nếu chỉ dùng giá trị gốc (không abs), feature có correlation −0.55 sẽ bị xếp thấp hơn feature có correlation +0.3, dù thực ra −0.55 mạnh hơn.


---

## 🔧 Phần B — Feature Engineering & Preprocessing (Q6–Q10)

---

### ❓ Câu 6: Feature `TotalSF = TotalBsmtSF + 1stFlrSF + 2ndFlrSF` thuộc loại feature engineering nào?

- A) Log feature — vì diện tích thường có phân phối lệch cần normalize
- B) Ordinal feature — vì tổng diện tích có thứ tự từ nhỏ đến lớn
- C) Combination feature — gộp nhiều cột liên quan thành 1 feature có ý nghĩa thực tế rõ hơn
- D) Binary feature — vì nhà có hoặc không có từng khu vực đó

---

> ✅ **Đáp án: C**
>
> `TotalSF` là **combination / domain knowledge feature** — thay vì để model tự tìm ra rằng 3 cột diện tích đều quan trọng, ta **gộp thẳng** thành tổng diện tích có ý nghĩa thực tế hơn (tổng diện tích sử dụng của toàn ngôi nhà).
>
> Đây là kỹ thuật mạnh với dữ liệu có domain knowledge rõ ràng:
> - Giảm số features → tránh overfitting
> - Feature mới trực tiếp phản ánh "kích thước nhà" — yếu tố ảnh hưởng nhất đến giá
> - Model học nhanh hơn và chính xác hơn

---

### ❓ Câu 7: `VarianceThreshold(threshold=0.01)` dùng để làm gì?

- A) Loại bỏ các features có tương quan cao với nhau (multicollinearity)
- B) Loại bỏ các features có phương sai quá thấp — gần như không thay đổi giữa các mẫu
- C) Chuẩn hóa các features về cùng một thang đo [0, 1]
- D) Chọn ra top K features có tương quan cao nhất với target

---

> ✅ **Đáp án: B**
>
> `VarianceThreshold` là **unsupervised feature selection** — loại bỏ các cột mà gần như tất cả các dòng có cùng giá trị (phương sai ≈ 0). Những cột như vậy không cung cấp thông tin phân biệt cho model.
>
> Ví dụ: Sau one-hot encoding, cột `Utilities_AllPub` có 99.9% giá trị = 1 → phương sai ≈ 0 → loại bỏ.
>
> **Lưu ý quan trọng:** Phải `fit` selector trên X_train rồi `transform` cả X_train lẫn X_test — không được fit trên X_test (data leakage)!

---

### ❓ Câu 8: Tại sao `StandardScaler` phải `fit` trên X_train và chỉ `transform` trên X_test?

- A) Vì fit trên X_test nghĩa là scaler tính mean/std từ test set → model "biết trước" thông tin tương lai → data leakage
- B) Vì X_test thường nhỏ hơn X_train nên kết quả fit sẽ không ổn định về mặt thống kê
- C) Vì scikit-learn chỉ cho phép gọi `fit()` một lần duy nhất trên một scaler object
- D) Vì X_test đã được Kaggle chuẩn hóa sẵn trước khi release, không cần fit thêm

---

> ✅ **Đáp án: A**
>
> Đây là lỗi **data leakage** — một trong những sai lầm phổ biến nhất trong ML.
>
> Nếu `fit_transform(X_test)`, scaler tính mean/std từ test set. Khi deploy thực tế, ta không có test set sẵn → phải dùng mean/std từ training. Kết quả: evaluation bị "ảo", không phản ánh hiệu suất thực.
>
> **Quy tắc vàng:** Mọi thứ (scaler, selector, encoder) chỉ được `fit` trên training data. Test data chỉ được `transform` theo thông số đã học từ train. Dùng `Pipeline` của sklearn để tự động hóa và tránh lỗi này.

---

### ❓ Câu 9: Với cột `ExterQual` có giá trị: `Po < Fa < TA < Gd < Ex`, ta nên dùng encoding nào?

- A) Target Encoding — vì chất lượng ngoại thất ảnh hưởng trực tiếp đến SalePrice
- B) One-Hot Encoding — vì đây là categorical variable cần vector hóa thành số
- C) Label Encoding ngẫu nhiên — model tree-based tự học thứ tự từ data
- D) Ordinal Encoding — vì các giá trị có thứ tự rõ ràng: Po(0) < Fa(1) < TA(2) < Gd(3) < Ex(4)

---

> ✅ **Đáp án: D**
>
> `ExterQual` là **ordinal variable** — các giá trị có thứ tự ý nghĩa rõ ràng. Ordinal encoding giữ lại thứ tự này, giúp Linear models hiểu "Ex tốt hơn Gd tốt hơn TA...".
>
> **Tại sao không dùng One-Hot (B)?**
> - Mất thông tin về thứ tự
> - Tăng số chiều không cần thiết (5 cột mới thay vì 1)
>
> **Khi nào dùng Target Encoding (A)?** Khi categorical variable có nhiều giá trị (high cardinality) như `Neighborhood` với 25+ giá trị. Với `ExterQual` chỉ có 5 giá trị có thứ tự, Ordinal Encoding đơn giản và hiệu quả hơn.

---

### ❓ Câu 10: Sau one-hot encoding có 300 features, sau `VarianceThreshold(0.01)` còn 220. Điều đó có nghĩa là gì?

- A) Model bị mất 80 features quan trọng → CV RMSE sẽ tăng (kém hơn)
- B) Nên tăng threshold lên 0.05 để giữ nhiều features hơn và tránh mất thông tin
- C) 80 features có phương sai < 0.01 — gần như constant — không cung cấp thông tin phân biệt → loại bỏ là đúng
- D) VarianceThreshold chọn ngẫu nhiên 220 trong 300 features mỗi lần chạy

---

> ✅ **Đáp án: C**
>
> 80 features bị loại có `Var < 0.01` — gần như mọi dòng đều có cùng giá trị. Những features như vậy **không cung cấp thêm thông tin** nào để phân biệt nhà đắt vs rẻ.
>
> Ví dụ điển hình: Sau one-hot encoding, cột `RoofStyle_Shed` có 99% giá trị = 0 (rất ít nhà mái lều) → phương sai ≈ 0.0099 < 0.01 → loại bỏ.
>
> **Kết quả thực tế:** Loại features nhiễu thường **cải thiện** CV RMSE, không làm giảm — model không bị phân tâm bởi các cột vô nghĩa.


---

## 🤖 Phần C — Model Training & Linear Models (Q11–Q15)

---

### ❓ Câu 11: Ridge Regression thêm penalty gì vào hàm loss?

- A) $L1 = \lambda \sum |w_i|$ — tổng giá trị tuyệt đối của các hệ số
- B) $L2 = \lambda \sum w_i^2$ — tổng bình phương của các hệ số
- C) $L1 + L2$ — kết hợp cả hai (đây là ElasticNet)
- D) Không thêm gì — Ridge chỉ là Linear Regression với learning rate nhỏ hơn

---

> ✅ **Đáp án: B**
>
> Ridge dùng **L2 regularization**: $\text{Loss} = \text{MSE} + \lambda \sum w_i^2$
>
> Penalty L2 **co nhỏ** tất cả hệ số về gần 0 nhưng **không bao giờ đúng bằng 0** — khác với Lasso (L1) có thể đưa về chính xác 0.
>
> Lợi ích:
> - Giảm overfitting khi có nhiều features
> - Xử lý multicollinearity hiệu quả
> - `alpha` (= λ) càng lớn → regularization càng mạnh → hệ số càng nhỏ
>
> **Ghi nhớ:** Lasso (A) → sparse model, tự chọn features. Ridge (B) → giữ tất cả features nhưng co nhỏ. ElasticNet (C) → kết hợp cả hai.

---

### ❓ Câu 12: `RidgeCV(alphas=[1, 10, 50, 100])` làm gì khác với `Ridge(alpha=50)`?

- A) RidgeCV lấy trung bình dự đoán của 4 Ridge models với 4 alpha khác nhau
- B) RidgeCV và Ridge(alpha=50) cho kết quả giống hệt nhau nếu alpha=50 là tối ưu — chỉ khác cách viết
- C) RidgeCV chạy 4 threads song song, mỗi thread xử lý một giá trị alpha
- D) RidgeCV tự động tìm alpha tối ưu từ danh sách bằng cross-validation nội bộ — không cần chỉ định thủ công

---

> ✅ **Đáp án: D**
>
> `RidgeCV` thực hiện **leave-one-out cross-validation** (hoặc generalized CV) để tự động chọn alpha tối ưu từ danh sách cho trước. Sau khi fit, `ridge.alpha_` cho biết giá trị được chọn.
>
> **Lợi ích:** Tiết kiệm công sức tuning thủ công. RidgeCV có thuật toán nội bộ rất hiệu quả — nhanh hơn nhiều so với GridSearchCV thông thường.
>
> **Sau khi `fit`**, model đã được train lại trên **toàn bộ** training data với alpha tối ưu → sẵn sàng dự đoán ngay.

---

### ❓ Câu 13: Model có Train RMSE = 0.08 nhưng CV RMSE = 0.14. Model đang gặp vấn đề gì?

- A) Overfitting — model học thuộc training data (kể cả nhiễu), không generalize tốt sang data mới
- B) Underfitting — model quá đơn giản, cần thêm features hoặc dùng model phức tạp hơn
- C) Data leakage — thông tin từ test set bị rò vào training, kết quả CV bị ảo
- D) Bình thường — Train RMSE luôn thấp hơn CV RMSE, khoảng cách 0.06 là chấp nhận được

---

> ✅ **Đáp án: A**
>
> Khoảng cách lớn giữa Train RMSE (0.08) và CV RMSE (0.14) là dấu hiệu điển hình của **overfitting**:
> - Model "học thuộc" training data, kể cả nhiễu ngẫu nhiên
> - Không tổng quát hóa tốt sang data mới
>
> **Cách khắc phục:**
> - Ridge/Lasso: tăng `alpha`
> - Random Forest: tăng `min_samples_leaf`, giảm `max_depth`
> - XGBoost: giảm `learning_rate`, giảm `max_depth`, tăng `reg_alpha`
> - Chung: thu thập thêm data, giảm số features

---

### ❓ Câu 14: `KFold(n_splits=5, shuffle=True, random_state=42)` — lý do chính cần `shuffle=True` là gì?

- A) Vì không shuffle thì các fold sẽ có kích thước không bằng nhau
- B) Vì data thường sắp xếp theo thứ tự (ID, thời gian...) — shuffle đảm bảo mỗi fold có phân phối đại diện cho toàn bộ dataset
- C) Vì shuffle giúp model hội tụ nhanh hơn trong quá trình training từng fold
- D) Vì `random_state=42` yêu cầu `shuffle=True` mới có tác dụng reproducible

---

> ✅ **Đáp án: B** *(D cũng đúng về mặt kỹ thuật — nhưng B là lý do chính)*
>
> Nếu data sắp xếp theo thứ tự (nhà rẻ trước, nhà đắt sau), không shuffle sẽ tạo ra các fold thiên lệch — fold 1 toàn nhà rẻ, fold 5 toàn nhà đắt → CV RMSE không phản ánh đúng hiệu suất thực.
>
> `shuffle=True` trộn ngẫu nhiên trước khi chia fold → mỗi fold có phân phối tương đồng nhau.  
> `random_state=42` giúp kết quả tái lập được — chạy nhiều lần vẫn cho cùng kết quả.

---

### ❓ Câu 15: Lasso chọn được 80 features quan trọng (hệ số ≠ 0) từ 220 features ban đầu. Điều này gọi là gì?

- A) Feature extraction — tạo ra features mới từ tổ hợp các features cũ (như PCA)
- B) Dimensionality reduction — giảm số chiều dữ liệu bằng phép chiếu tuyến tính
- C) Data augmentation — tăng số lượng mẫu training bằng cách biến đổi data hiện có
- D) Feature selection — tự động chọn subset features quan trọng thông qua L1 regularization

---

> ✅ **Đáp án: D**
>
> Lasso (L1) có tính chất đặc biệt: đưa một số hệ số về **đúng bằng 0** → loại bỏ hoàn toàn các features đó. Đây là **automatic feature selection** — không cần tay chọn thủ công.
>
> **Tại sao L1 làm được, L2 (Ridge) không?**
> - L1 penalty (`|w|`) tạo góc nhọn tại w=0 → nghiệm tối ưu hay rơi vào góc → w=0 chính xác
> - L2 penalty (`w²`) trơn tại w=0 → nghiệm chỉ *gần* 0, không bao giờ = đúng 0
>
> Ứng dụng: Chạy Lasso để biết "features nào thực sự quan trọng" trước khi build model phức tạp hơn.


---

## 🏆 Phần D — Ensemble & Evaluation (Q16–Q20)

---

### ❓ Câu 16: Weighted Ensemble gán trọng số `w ∝ 1/RMSE`. Model A (RMSE=0.11) và B (RMSE=0.13). Trọng số là?

- A) A≈0.54, B≈0.46 — Model A có RMSE thấp hơn nên được tin tưởng hơn
- B) A=0.5, B=0.5 — chia đều vì chênh lệch RMSE giữa hai model không đáng kể
- C) A=1.0, B=0.0 — chỉ giữ model tốt nhất, loại bỏ model kém hơn
- D) A=0.13, B=0.11 — trọng số bằng RMSE của model kia (cross-assignment)

---

> ✅ **Đáp án: A**
>
> Tính toán cụ thể:
> - `inv_A = 1/0.11 ≈ 9.09` ;  `inv_B = 1/0.13 ≈ 7.69`
> - Tổng = 16.78
> - `w_A = 9.09 / 16.78 ≈ 0.542` ;  `w_B = 7.69 / 16.78 ≈ 0.458`
> - Blend = 0.542 × pred_A + 0.458 × pred_B
>
> **Tại sao không chọn C (chỉ dùng A)?** Ensemble luôn ổn định hơn 1 model đơn lẻ — model B dù kém hơn vẫn bắt được một số pattern mà A bỏ sót, giúp giảm variance của dự đoán cuối.

---

### ❓ Câu 17: Stacking Ensemble khác Weighted Blend ở điểm nào quan trọng nhất?

- A) Stacking lấy trung bình đơn giản (equal weight) của tất cả base model predictions
- B) Stacking chỉ hoạt động được với tree-based models (RF, XGBoost), không dùng với Linear models
- C) Stacking dùng meta-learner (model cấp 2) để học cách kết hợp tối ưu — không phải trọng số cố định
- D) Stacking luôn cho kết quả tốt hơn Weighted Blend trong mọi trường hợp và dataset

---

> ✅ **Đáp án: C**
>
> **Weighted Blend:** Trọng số cố định, tính dựa trên CV RMSE — đơn giản nhưng không linh hoạt.
>
> **Stacking:**
> 1. Train các base models (Ridge, RF, XGBoost) bằng cross-validation → tạo **out-of-fold predictions** (OOF)
> 2. Dùng OOF predictions làm features mới → train **meta-learner** (thường là Ridge đơn giản)
> 3. Meta-learner **tự học** trọng số tối ưu, thậm chí nắm được pattern phi tuyến giữa các model
>
> Stacking phức tạp hơn, dễ overfit hơn, nhưng khi data đủ lớn thường vượt trội Weighted Blend.

---

### ❓ Câu 18: Kaggle Public Leaderboard dùng ~50% test set. Tại sao không dùng 100%?

- A) Vì Kaggle không đủ server để tính điểm trên 100% test set cùng lúc trong thời gian thực
- B) Vì 50% test set chưa có nhãn — Kaggle bổ sung nhãn dần dần trong lúc competition còn mở
- C) Vì 50% còn lại được dùng để validate hệ thống chấm điểm nội bộ của Kaggle
- D) Vì 50% được giữ lại để tính **Private Score** — ngăn chặn "overfit" Public LB qua việc submit nhiều lần

---

> ✅ **Đáp án: D**
>
> Nếu Public Score dùng 100% test, người tham gia có thể **reverse-engineer nhãn** bằng cách submit nhiều lần và quan sát điểm thay đổi → gian lận.
>
> **Cơ chế Kaggle:**
> - **Public LB (~50%):** Hiển thị trong lúc competition mở → dùng để theo dõi tiến độ
> - **Private LB (100%):** Chỉ hiện sau khi competition kết thúc → kết quả chính thức xếp hạng
>
> Bài học: Đừng "overfit" vào Public Score bằng cách chọn submission dựa trên LB — "Public LB shakeup" là hiện tượng rank thay đổi mạnh khi Private LB được công bố.

---

### ❓ Câu 19: RMSE = 0.12 trên log(SalePrice) tương ứng với sai số bao nhiêu phần trăm trên giá gốc?

- A) Đúng 12% — vì RMSE = 0.12 và log ở đây chỉ là đơn vị tính
- B) Khoảng 12.75% — vì RMSE trên log space xấp xỉ MAPE trên giá gốc (e^0.12 ≈ 1.1275)
- C) $12,000 USD — vì RMSE đo sai số tuyệt đối trung bình
- D) Không thể tính được — cần biết giá trị SalePrice cụ thể của từng nhà

---

> ✅ **Đáp án: B**
>
> Khi target là `log1p(y)`, RMSE trên log space xấp xỉ **MAPE (Mean Absolute Percentage Error)** trên giá gốc.
>
> Tính toán: Nếu `log_pred = log_true ± 0.12` thì:
> - `pred/true = e^(±0.12) ≈ 0.887 đến 1.127`
> - Sai số ≈ ±12.75% so với giá thực
>
> **Ý nghĩa thực tế:** Nhà giá $200,000 → model dự đoán trong khoảng $177K–$225K. Với bài toán định giá BĐS, mức sai số ±13% là chấp nhận được cho ML model cơ bản.

---

### ❓ Câu 20: Bước nào sau đây KHÔNG phải cách cải thiện RMSE hợp lệ?

- A) Thêm feature tương tác `OverallQual × GrLivArea` dựa trên domain knowledge
- B) Dùng LightGBM bổ sung vào ensemble để tăng diversity
- C) Submit thật nhiều lần, chọn submission có Public Score tốt nhất để nộp cuối
- D) Target encoding cho `Neighborhood` dựa trên median SalePrice của từng khu

---

> ✅ **Đáp án: C**
>
> Đây là **anti-pattern** — còn gọi là *"Public LB overfitting"* hay *"submission fishing"*.
>
> Khi bạn submit 100 lần và chọn cái có Public Score tốt nhất, bạn đang **overfit vào 50% Public test set** chứ không thực sự cải thiện model. Kết quả: Private Score thường tệ hơn nhiều so với kỳ vọng.
>
> **Ba đáp án còn lại đều hợp lệ:**
> - A: Interaction feature có domain knowledge → thường giúp Linear models rất nhiều
> - B: Thêm model đa dạng vào ensemble → giảm variance tổng thể
> - D: Target encoding nắm bắt effect địa lý lên giá nhà
>
> **Quy tắc vàng:** Cải thiện model dựa trên **CV offline**, không dựa trên Public Leaderboard.


---

## 📊 Bảng Đáp Án Nhanh

| Câu | Đáp án | Chủ đề |
|-----|--------|--------|
| Q1  | **A** | log1p giúp phân phối gần chuẩn |
| Q2  | **C** | Fill LotFrontage theo Neighborhood |
| Q3  | **A** | Outlier: diện tích lớn nhưng giá thấp |
| Q4  | **B** | Correlation đo mức độ liên hệ tuyến tính |
| Q5  | **D** | Abs correlation đo sức mạnh bất kể chiều |
| Q6  | **C** | TotalSF là combination feature |
| Q7  | **B** | VarianceThreshold loại features phương sai thấp |
| Q8  | **A** | Fit trên test set → data leakage |
| Q9  | **D** | ExterQual dùng ordinal encoding |
| Q10 | **C** | 80 features constant → loại bỏ đúng |
| Q11 | **B** | Ridge dùng L2 penalty |
| Q12 | **D** | RidgeCV tự tìm alpha tối ưu |
| Q13 | **A** | Train–CV gap lớn → overfitting |
| Q14 | **B** | Shuffle đảm bảo phân phối đại diện |
| Q15 | **D** | Lasso = automatic feature selection |
| Q16 | **A** | Model RMSE thấp hơn → trọng số cao hơn |
| Q17 | **C** | Stacking dùng meta-learner |
| Q18 | **D** | 50% giữ lại cho Private Score |
| Q19 | **B** | RMSE 0.12 ≈ sai số 12.75% giá gốc |
| Q20 | **C** | Submit fishing là anti-pattern |

---

## 🎓 Tổng Kết Khóa Học — 9 Buổi Dự Đoán Giá Nhà

```
Buổi 1  ✅  Hiểu bài toán, lộ trình, Kaggle competition
Buổi 2  ✅  Setup môi trường, pandas, numpy, matplotlib
Buổi 3  ✅  EDA — phân phối, correlation, missing values, outliers
Buổi 4  ✅  Data Cleaning — fill missing, xóa outlier, encode
Buổi 5  ✅  Feature Engineering — TotalSF, TotalBath, interaction, log features
Buổi 6  ✅  Model Training — Ridge, Lasso, RF, XGBoost, Stacking, Averaging
Buổi 7  ✅  Model Evaluation — Train/CV RMSE, learning curve, residuals, chiến lược
Buổi 8  ✅  Submission — rebuild pipeline, weighted blend, validate, nộp Kaggle
Buổi 9  🎓  Quiz Tốt Nghiệp — 20 câu ôn tập toàn bộ kiến thức
```

---

### 🏅 Kết Quả Của Bạn

| Số câu đúng | Nhận xét |
|------------|---------|
| **18 – 20** | 🥇 Xuất sắc — Nắm vững toàn bộ kiến thức! |
| **14 – 17** | 🥈 Tốt — Hiểu cơ bản, ôn lại phần chưa chắc |
| **10 – 13** | 🥉 Trung bình — Nên xem lại notebook các buổi tương ứng |
| **< 10**   | 📚 Cần ôn tập thêm — Đọc lại từ Buổi 3–7 |

---

> ## 🎉 Chúc mừng bạn đã hoàn thành khóa học!
>
> Bạn đã đi qua **toàn bộ pipeline Machine Learning thực tế**:  
> **Raw Data → EDA → Cleaning → Features → Training → Evaluation → Submission**
>
> Đây là nền tảng vững chắc để tiếp tục với:
> - 🔥 **LightGBM / CatBoost** — boosting nhanh hơn XGBoost
> - 🧠 **Neural Networks** — cho tabular data (TabNet, MLP)
> - ⚙️ **AutoML** — tự động hóa toàn bộ pipeline
> - 📦 **MLOps** — deploy model vào production thực tế
